In [3]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings 

In [5]:
# setting an API Key
os.environ["GOOGLE_API_KEY"] = "AQ.....-CTDw"

In [6]:
#load the PDF
from langchain_community.document_loaders import PyPDFLoader
# Windows Absolute Path
loader = PyPDFLoader("Hundreddaysml.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

C:\Users\ashim\AppData\Local\Temp\ipykernel_127896\246884363.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_docs"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
#model & chat prompt template
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are helpful assistant. Answer the question using only the provided context"),
    ("human","Context:\n {context}\n\n Question:\n{question}")
])

In [9]:
question = "What services does the company provide?"
results = retriever.invoke(question)
context_text = "\n\n".join(
    doc.page_content for doc in results)


In [10]:
#format the prompt using chatprompt template
formatted_messages = prompt_template.format_messages(
    context=context_text,
    question=question
)

In [ ]:
test = llm.invoke("Explain machine learning in one sentence.")
print(test.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [ ]:
#Invoke LLM
response = llm.invoke(formatted_messages)
print(response.content)